# Bag-of-Words: turning text into a vector of word counts

_Feature Engineering_

**Count how many times each vocabulary word appears in a document, ignore the order.**

A machine learning model eats numbers, not sentences. So the first job in any text task is
       to turn a document into a vector of numbers. Bag-of-words (BoW) is the simplest honest way to
       do that: list every word that appears anywhere in your collection (the vocabulary), then
       describe each document by how many times each of those words shows up in it.

       The name says it all. You dump the document's words into a bag and shake it. You keep which
       words are present and how many of each — but you throw away the order and the grammar.
       "the food was good" and "good was the food" land in the exact same bag.

---

This notebook is a practice scaffold for the **Bag-of-Words: turning text into a vector of word counts** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## See it for yourself: the problem, then the fix

Run this cell. It plots the **raw** data and reproduces the problem it causes, then plots the **engineered** data and shows the fix — with before/after numbers.

In [ ]:
# Lesson: Bag-of-words — turn raw text into a numeric word-count vector.
# A machine-learning model does math on NUMBERS. It cannot do math on a Python
# string like "great movie". So if you hand a classifier a list of raw text
# strings, fitting it FAILS — there is nothing to add or multiply.
# FIX: bag-of-words. Build a vocabulary of every word in the corpus, then turn
# each sentence into a vector of counts: how many times each vocab word appears.
# Now every sentence is a row of numbers, and the SAME classifier trains fine.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

# ---- STEP 1: Load real data --------------------------------------------------
# A small REAL inline corpus of short movie reviews. No downloads.
# label 1 = positive review, label 0 = negative review. 10 sentences, 2 classes.
texts = [
    "great movie loved it",          # 1
    "what a fantastic film",         # 1
    "loved the acting wonderful",    # 1
    "an amazing and great story",    # 1
    "wonderful film truly amazing",  # 1
    "terrible movie hated it",       # 0
    "what an awful film",            # 0
    "boring and terrible acting",    # 0
    "a dull and awful story",        # 0
    "hated this boring film",        # 0
]
y = np.array([1, 1, 1, 1, 1, 0, 0, 0, 0, 0])

# Hold out 2 brand-new sentences the model never sees during training.
test_texts = ["a great wonderful film", "awful and boring movie"]
test_y = np.array([1, 0])  # positive, negative

# ---- STEP 2: Visualize the PURE (raw) data -----------------------------------
# "Raw data" here is just strings. The only number we can read off raw text is
# its length in characters — which has NOTHING to do with sentiment. Plot it to
# show raw text gives the model no usable signal.
char_len = np.array([len(t) for t in texts])
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
colors = ["#27ae60" if lab == 1 else "#c0392b" for lab in y]
ax[0].bar(range(len(texts)), char_len, color=colors)
ax[0].set_title("STEP 2: raw text has no usable numbers\n(bar = #characters; "
                "green=positive, red=negative — length is meaningless)")
ax[0].set_xlabel("review index"); ax[0].set_ylabel("characters in string")

# ---- STEP 3: Reproduce the PROBLEM on the raw data ---------------------------
# Try to fit a classifier directly on the list of raw STRINGS. It cannot turn
# "great movie loved it" into numbers, so this raises an error. We catch it.
print("STEP 3 PROBLEM  feeding raw text strings straight to the model:")
raw_worked = False
try:
    LogisticRegression().fit(texts, y)   # texts is a list of strings, not numbers
    raw_worked = True
    print("  (unexpectedly succeeded)")
except Exception as e:
    print(f"  -> FAILED with {type(e).__name__}: {e}")
    print("  The model needs a numeric matrix; a list of strings is not one.")

# ---- STEP 4: Apply bag-of-words, then visualize the engineered data ----------
# CountVectorizer learns the vocabulary from the training corpus and turns each
# sentence into a row of word counts. Text -> a matrix of numbers.
vec = CountVectorizer()
X_train = vec.fit_transform(texts).toarray()   # shape (n_docs, n_vocab_words)
vocab = vec.get_feature_names_out()
print(f"\nSTEP 4 FIX      bag-of-words made a {X_train.shape[0]} x {X_train.shape[1]} "
      f"document-term matrix (rows=reviews, cols=words). Text is now numbers.")

# Heatmap of the document-term matrix: words on x, reviews on y, color = count.
im = ax[1].imshow(X_train, aspect="auto", cmap="Greens")
ax[1].set_xticks(range(len(vocab))); ax[1].set_xticklabels(vocab, rotation=90, fontsize=8)
ax[1].set_yticks(range(len(texts)))
ax[1].set_yticklabels([f"{i} ({'pos' if y[i] else 'neg'})" for i in range(len(texts))], fontsize=8)
ax[1].set_title("STEP 4: document-term matrix (bag-of-words)\ntext became a grid of word counts")
fig.colorbar(im, ax=ax[1], label="word count")
plt.tight_layout(); plt.show()

# ---- STEP 5: Show the FIX ----------------------------------------------------
# Same classifier, now on the numeric bag-of-words matrix. It trains, and it
# classifies the held-out sentences correctly.
clf = LogisticRegression()
clf.fit(X_train, y)                       # works now: X_train is numbers
X_test = vec.transform(test_texts).toarray()  # reuse the SAME vocabulary
pred = clf.predict(X_test)
acc = (pred == test_y).mean()
for t, p in zip(test_texts, pred):
    print(f"  held-out: {t!r:30s} -> predicted {'positive' if p else 'negative'}")

raw_score = 0.0 if not raw_worked else float("nan")  # raw could not even run
print(f"\nPROBLEM (raw text): cannot fit, accuracy = {raw_score:.3f}   ->   "
      f"FIX (bag-of-words): accuracy = {acc:.3f}")

## Reference implementation — scikit-learn + pandas

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

# Yelp reviews dataset (Chapter 3 of Zheng & Casari).
# Get it from the book's repo: github.com/alicezheng/feature-engineering-book
# (the Yelp academic dataset). Each line of the JSON is one review object.
reviews = pd.read_json('yelp_academic_dataset_review.json', lines=True)
review_texts = reviews['text']          # the column of raw review strings

# --- Bag-of-Words ---
bow = CountVectorizer()                 # lowercases & tokenizes on word boundaries
X = bow.fit_transform(review_texts)     # SPARSE document-term matrix: one row per review,
                                        # one column per vocabulary word, entry = word count

vocab = bow.get_feature_names_out()     # the vocabulary: column index -> word
print("documents x vocabulary:", X.shape)     # e.g. (10000, 23000)
print("first 10 vocabulary words:", vocab[:10])

# X is sparse -- keep it sparse; .toarray() on a real corpus would be gigabytes.
print("stored (non-zero) entries:", X.nnz)
print("sparsity:", 1 - X.nnz / (X.shape[0] * X.shape[1]))   # almost all zeros

# Each document is now a vector of word counts -- a point in bag-of-words space.
# Feed X straight into a linear classifier to predict, e.g., the review's star rating:
#   from sklearn.linear_model import LogisticRegression
#   LogisticRegression().fit(X, reviews['stars'])

## Visualize the data & results

_On four tiny real review-style sentences, what does the bag-of-words document-term matrix look like, which words are most common, and can it tell 'not good' from 'good'?_

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

# Four short, real review-style sentences (inline -- no download needed).
docs = [
    "the food was good and the service was great",
    "the service was slow but the food was good",
    "great food great service great value",
    "the food was not good and the service was slow",
]

bow = CountVectorizer()                 # same call the book uses
X = bow.fit_transform(docs)             # sparse document-term matrix
vocab = bow.get_feature_names_out()
A = X.toarray()                         # tiny here, so densify for display

print("vocabulary:", list(vocab))
print("matrix shape (docs x words):", A.shape)   # (4, 11)
print(A)

# Total count of each word across the whole corpus (the summed bag):
totals = A.sum(axis=0)
order = np.argsort(-totals)
for i in order:
    print(f"{vocab[i]:>8}: {totals[i]}")
# -> the:6  was:6  food:4  great:4  service:4  good:3  and:2  slow:2  but:1  not:1  value:1

# Order is lost: doc 1 ("...good...") vs doc 4 ("...not good...") differ
# only in the 'not' (+1) and 'and' columns -- the negation is a single stray count.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A sentiment classifier built on bag-of-words rates the review "the food was not good" almost the same as "the food was good". Why, and what is the standard fix?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Write both reviews as bags of word counts and compare them. — _BoW keeps only how many times each word appears, so both reviews share "the", "food", "was", "good"; they differ only in the single extra word "not"._
- Notice the word "not" sits in its own column, unattached to "good". — _Because order is discarded, BoW cannot represent that "not" modifies "good" — the negation is just one more isolated count._
- Add bag-of-n-grams: count 2-word sequences (bigrams) as features too. — _Then "not good" becomes a single feature distinct from "good", restoring the negation the model needs._

**Answer:** BoW throws away word order, so "not good" and "good" share almost all the same word counts and look nearly identical — the negation in "not" floats free in its own column. The standard fix is n-grams (bag-of-n-grams): counting "not good" as its own feature. This is exactly why Chapter 3 introduces n-grams right after bag-of-words.

</details>

**Problem 2.** You run CountVectorizer on 50,000 Yelp reviews and the vocabulary has 28,000 words. Why is calling .toarray() on the result a bad idea, and what dominates the top counts?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Estimate the dense size: 50,000 rows by 28,000 columns of integers. — _That is 1.4 billion entries — gigabytes of memory — even though almost every entry is 0._
- Recall that each review uses only a few dozen distinct words. — _So each row has a few dozen non-zeros out of 28,000 columns: the matrix is extremely sparse and should stay a sparse matrix._
- Sum each column across all reviews and look at the largest totals. — _The biggest counts are function words like "the", "and", "was" — frequent but nearly meaningless._

**Answer:** .toarray() would expand a mostly-zero sparse matrix into a dense $50000\times28000$ block — gigabytes for almost no information; keep it sparse. And the top raw counts are dominated by common stop words ("the", "and", "was"), which is precisely why you remove stop words or switch to tf-idf weighting.

</details>

**Problem 3.** Two short reviews are $d_2=$ "good food" and $d_3=$ "bad service" over the vocabulary [bad, food, good, service]. What is their cosine similarity in bag-of-words space, and what does it tell you?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Write the vectors: $\mathbf{x}_2=[0,1,1,0]$, $\mathbf{x}_3=[1,0,0,1]$. — _Each coordinate is the count of that vocabulary word in the review._
- Compute the dot product $\mathbf{x}_2\cdot\mathbf{x}_3 = 0\cdot1+1\cdot0+1\cdot0+0\cdot1 = 0$. — _The two reviews share no vocabulary word, so every product term is 0._
- Cosine similarity $= \dfrac{\mathbf{x}_2\cdot\mathbf{x}_3}{\lVert\mathbf{x}_2\rVert\,\lVert\mathbf{x}_3\rVert} = \dfrac{0}{\sqrt2\,\sqrt2}=0$. — _A zero dot product means the vectors are orthogonal — at a right angle in BoW space._

**Answer:** Their cosine similarity is 0 — the vectors are orthogonal. Sharing no words puts two documents at a right angle in bag-of-words space, the maximum possible dissimilarity. This is the geometry the book stresses: word overlap becomes nearness, and disjoint vocabularies become orthogonality.

</details>